In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

save_dir_png = ROOT / 'outputs/figures/anomaly_minmax'
os.makedirs(save_dir_png, exist_ok=True)

In [2]:
def fetch_daily(meter_urn: str, measurement: str) -> pd.DataFrame:
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    df['day'] = pd.to_datetime(df['day'])
    return df

In [3]:
# ============================================================
# 1. V.Z81 W_in 극단값 (-5.5조) 확인
# ============================================================
df = fetch_daily('V.Z81', 'W_in')

# 이상값 날짜 출력
threshold = -1_000_000  # 100만 이하 음수를 이상값으로 간주
anomaly = df[df['min_val'] < threshold]
print(f'V.Z81 W_in 이상값 날짜 ({len(anomaly)}건):')
print(anomaly[['day', 'min_val', 'max_val']].to_string())

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['W_in 일별 min', 'W_in 일별 max'],
                    shared_xaxes=True)

fig.add_trace(go.Scatter(x=df['day'], y=df['min_val'], mode='lines', name='min'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['day'], y=df['max_val'], mode='lines', name='max'), row=2, col=1)

fig.update_layout(title='V.Z81 W_in 일별 min/max (2018~2023)', height=600, template='plotly_white')
fig.write_image(str(save_dir_png / 'V.Z81_W_in_daily.png'), width=1400, height=600)
print('V.Z81 W_in 저장 완료')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z81 W_in 이상값 날짜 (2188건):
            day       min_val       max_val
3    2018-01-04 -1.133069e+09 -1.539988e+07
4    2018-01-05 -1.579243e+09 -9.211151e+08
5    2018-01-06 -1.579208e+09 -1.363633e+09
6    2018-01-07 -1.800692e+09 -1.363517e+09
7    2018-01-08 -1.593294e+09 -1.367392e+09
8    2018-01-09 -2.242245e+09 -1.577547e+09
9    2018-01-10 -3.570738e+09 -2.032024e+09
10   2018-01-11 -3.791179e+09 -3.361417e+09
11   2018-01-12 -4.230898e+09 -3.790817e+09
12   2018-01-13 -4.889087e+09 -4.230670e+09
13   2018-01-14 -4.888974e+09 -4.677247e+09
14   2018-01-15 -5.329060e+09 -4.680885e+09
15   2018-01-16 -5.328944e+09 -5.117753e+09
16   2018-01-17 -5.547014e+09 -5.328668e+09
17   2018-01-18 -5.546717e+09 -5.546717e+09
18   2018-01-19 -5.546717e+09 -5.546717e+09
19   2018-01-20 -5.546717e+09 -5.546717e+09
20   2018-01-21 -5.546717e+09 -5.546716e+09
21   2018-01-22 -5.546716e+09 -5.546716e+09
22   2018-01-23 -5.546716e+09 -5.546716e+09
23   2018-01-24 -5.546716e+09 -5.546715e+09
24   

In [4]:
# ============================================================
# 2. H1.K15 Trl -151도 확인
# ============================================================
df = fetch_daily('H1.K15', 'Trl')

# 이상값 날짜 출력
threshold = -10  # -10도 이하를 이상값으로 간주
anomaly = df[df['min_val'] < threshold]
print(f'H1.K15 Trl 이상값 날짜 ({len(anomaly)}건):')
print(anomaly[['day', 'min_val', 'max_val']].to_string())

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['Trl 일별 min', 'Trl 일별 max'],
                    shared_xaxes=True)

fig.add_trace(go.Scatter(x=df['day'], y=df['min_val'], mode='lines', name='min'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['day'], y=df['max_val'], mode='lines', name='max'), row=2, col=1)

fig.update_layout(title='H1.K15 Trl 일별 min/max (2018~2023)', height=600, template='plotly_white')
fig.write_image(str(save_dir_png / 'H1.K15_Trl_daily.png'), width=1400, height=600)
print('H1.K15 Trl 저장 완료')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K15 Trl 이상값 날짜 (2건):
          day     min_val    max_val
53 2018-02-23 -151.808333  13.333333
77 2018-03-19  -49.500000  13.033333
H1.K15 Trl 저장 완료


In [5]:
# ============================================================
# 3. H4.Z51 WQ 전체 음수 고착 확인
# ============================================================
df = fetch_daily('H4.Z51', 'WQ')

print(f'H4.Z51 WQ 전체 기간 min: {df["min_val"].min():.2f}, max: {df["max_val"].max():.2f}')
print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['WQ 일별 min', 'WQ 일별 max'],
                    shared_xaxes=True)

fig.add_trace(go.Scatter(x=df['day'], y=df['min_val'], mode='lines', name='min'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['day'], y=df['max_val'], mode='lines', name='max'), row=2, col=1)

fig.update_layout(title='H4.Z51 WQ 일별 min/max (2018~2023)', height=600, template='plotly_white')
fig.write_image(str(save_dir_png / 'H4.Z51_WQ_daily.png'), width=1400, height=600)
print('H4.Z51 WQ 저장 완료')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H4.Z51 WQ 전체 기간 min: -158435.62, max: -10815.68
양수 구간 존재 여부: False
H4.Z51 WQ 저장 완료


In [6]:
# ============================================================
# 4. H1.K15 Tvl, Tdiff 확인
# ============================================================
for measurement, threshold, label in [
    ('Tvl', -10, '-10도 이하'),
    ('Tdiff', -15000, '-15000 mK 이하'),
]:
    df = fetch_daily('H1.K15', measurement)
    anomaly = df[df['min_val'] < threshold]
    print(f'H1.K15 {measurement} 이상값 ({label}) {len(anomaly)}건:')
    print(anomaly[['day', 'min_val', 'max_val']].to_string())
    print()

    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=[f'{measurement} 일별 min', f'{measurement} 일별 max'],
                        shared_xaxes=True)
    fig.add_trace(go.Scatter(x=df['day'], y=df['min_val'], mode='lines', name='min'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['day'], y=df['max_val'], mode='lines', name='max'), row=2, col=1)
    fig.update_layout(title=f'H1.K15 {measurement} 일별 min/max (2018~2023)', height=600, template='plotly_white')
    fig.write_image(str(save_dir_png / f'H1.K15_{measurement}_daily.png'), width=1400, height=600)
    print(f'H1.K15 {measurement} 저장 완료')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K15 Tvl 이상값 (-10도 이하) 2건:
          day    min_val    max_val
53 2018-02-23 -43.758333  22.025000
77 2018-03-19 -42.800000  23.491667

H1.K15 Tvl 저장 완료


/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K15 Tdiff 이상값 (-15000 mK 이하) 282건:
            day       min_val       max_val
53   2018-02-23 -18962.583333  11531.750000
226  2018-08-15 -15147.750000  -8134.333333
243  2018-09-01 -15520.583333  -7723.666667
244  2018-09-02 -15616.500000  -7661.666667
277  2018-10-05 -16355.833333  -8092.333333
278  2018-10-06 -16218.500000  -8119.333333
280  2018-10-08 -16243.250000  -7949.166667
281  2018-10-09 -16410.833333  -7281.666667
282  2018-10-10 -16469.583333  -7881.500000
283  2018-10-11 -16462.583333  -8057.000000
479  2019-04-25 -15142.250000  -7754.500000
502  2019-05-18 -15395.833333  -7523.666667
504  2019-05-20 -16193.611111  -7510.666667
517  2019-06-02 -18484.166667  -7956.666667
526  2019-06-11 -18478.166667  -7676.750000
527  2019-06-12 -18011.666667  -3095.666667
542  2019-06-27 -18731.166667  -9349.166667
543  2019-06-28 -18290.500000  -9142.666667
549  2019-07-04 -16529.599727  -7091.000000
555  2019-07-10 -17366.250000  -8671.500000
560  2019-07-15 -19064.000000  -8020.6

In [7]:
# ============================================================
# 5. H1.K16 Tdiff 확인
# ============================================================
df = fetch_daily('H1.K16', 'Tdiff')
threshold = -15000
anomaly = df[df['min_val'] < threshold]
print(f'H1.K16 Tdiff 이상값 ({len(anomaly)}건):')
print(anomaly[['day', 'min_val', 'max_val']].to_string())

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=['Tdiff 일별 min', 'Tdiff 일별 max'],
                    shared_xaxes=True)
fig.add_trace(go.Scatter(x=df['day'], y=df['min_val'], mode='lines', name='min'), row=1, col=1)
fig.add_trace(go.Scatter(x=df['day'], y=df['max_val'], mode='lines', name='max'), row=2, col=1)
fig.update_layout(title='H1.K16 Tdiff 일별 min/max (2018~2023)', height=600, template='plotly_white')
fig.write_image(str(save_dir_png / 'H1.K16_Tdiff_daily.png'), width=1400, height=600)
print('H1.K16 Tdiff 저장 완료')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.K16 Tdiff 이상값 (6건):
            day       min_val       max_val
970  2021-07-23 -18231.583333   1517.416667
971  2021-07-24 -21713.500000 -13540.166667
972  2021-07-25 -24456.083333 -21821.416667
973  2021-07-26 -23437.166667  -9082.750000
1509 2023-01-13 -15280.916667  -2662.750000
1547 2023-02-20 -15472.333333  -2176.750000
H1.K16 Tdiff 저장 완료


In [8]:
# ============================================================
# 6. H1.Z15 I1, I2, I3, P 확인
# ============================================================
for measurement, threshold, label in [
    ('I1', -100, '-100A 이하'),
    ('I2', -100, '-100A 이하'),
    ('I3', -100, '-100A 이하'),
    ('P', -50000, '-50000W 이하'),
]:
    df = fetch_daily('H1.Z15', measurement)
    anomaly = df[df['min_val'] < threshold]
    print(f'H1.Z15 {measurement} 이상값 ({label}) {len(anomaly)}건:')
    if len(anomaly) > 0:
        print(anomaly[['day', 'min_val', 'max_val']].head(20).to_string())
    print()

    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=[f'{measurement} 일별 min', f'{measurement} 일별 max'],
                        shared_xaxes=True)
    fig.add_trace(go.Scatter(x=df['day'], y=df['min_val'], mode='lines', name='min'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['day'], y=df['max_val'], mode='lines', name='max'), row=2, col=1)
    fig.update_layout(title=f'H1.Z15 {measurement} 일별 min/max (2018~2023)', height=600, template='plotly_white')
    fig.write_image(str(save_dir_png / f'H1.Z15_{measurement}_daily.png'), width=1400, height=600)
    print(f'H1.Z15 {measurement} 저장 완료')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z15 I1 이상값 (-100A 이하) 2145건:
          day     min_val    max_val
0  2018-01-01 -132.751293 -51.181356
1  2018-01-02 -130.595435 -17.093455
3  2018-01-04 -128.499162  25.491789
4  2018-01-05 -134.021659  53.807240
5  2018-01-06 -127.436856  -7.656960
7  2018-01-08 -112.800079  30.796221
8  2018-01-09 -123.262736  30.279733
9  2018-01-10 -127.533437  88.566864
10 2018-01-11 -126.266368  83.191064
11 2018-01-12 -106.060155  75.898801
13 2018-01-14 -111.615975  -5.992971
14 2018-01-15 -127.987028   3.050447
15 2018-01-16 -101.164265  -0.671780
16 2018-01-17 -109.754900  61.207626
17 2018-01-18 -105.636935   9.882893
18 2018-01-19 -109.276769  42.676267
19 2018-01-20 -101.549448  14.596168
21 2018-01-22 -113.765931  24.656242
23 2018-01-24 -100.544871   5.928158
24 2018-01-25 -102.157002  -4.708321

H1.Z15 I1 저장 완료


/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z15 I2 이상값 (-100A 이하) 2086건:
          day     min_val     max_val
70 2018-03-12 -153.776977   14.302750
80 2018-03-22 -108.843353   98.035338
81 2018-03-23 -115.336304  -72.749167
82 2018-03-24 -104.962340  -97.066969
83 2018-03-25 -106.272210  -94.990165
84 2018-03-26 -113.157621  -73.913285
85 2018-03-27 -119.874639  -67.787821
86 2018-03-28 -108.552957  -89.035814
87 2018-03-29 -145.342105  -83.280128
88 2018-03-30 -145.678447 -103.027966
89 2018-03-31 -146.207270  -98.638433
90 2018-04-01 -156.230364  -98.390141
91 2018-04-02 -152.780854  -93.848923
92 2018-04-03 -229.093336  -89.583163
93 2018-04-04 -254.728538  -71.152816
94 2018-04-05 -126.512208  -70.389424
95 2018-04-06 -149.355267  -64.295974
96 2018-04-07 -156.511460  -96.904172
97 2018-04-08 -124.727946  -94.239748
98 2018-04-09 -148.535488  -75.871918

H1.Z15 I2 저장 완료


/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z15 I3 이상값 (-100A 이하) 2128건:
          day     min_val     max_val
57 2018-02-27 -161.954963  -25.257905
58 2018-02-28 -177.243679 -111.444281
59 2018-03-01 -173.391794 -113.632618
60 2018-03-02 -127.410958  -92.268981
61 2018-03-03 -116.722435 -110.980629
62 2018-03-04 -113.540319 -110.114644
63 2018-03-05 -157.882070 -110.518177
64 2018-03-06 -152.966057 -110.342524
65 2018-03-07 -156.501834 -109.926587
66 2018-03-08 -154.789521 -111.285463
67 2018-03-09 -154.489207 -107.505393
68 2018-03-10 -109.862705 -105.538609
69 2018-03-11 -111.021331 -106.222092
70 2018-03-12 -125.371560  -21.504261
71 2018-03-13 -132.006742  -27.865998
72 2018-03-14 -174.358119 -110.015511
73 2018-03-15 -161.577718 -112.965016
74 2018-03-16 -177.790991 -121.553740
75 2018-03-17 -123.000493 -109.462918
76 2018-03-18 -123.015750 -109.757196

H1.Z15 I3 저장 완료


/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z15 P 이상값 (-50000W 이하) 1019건:
          day       min_val       max_val
0  2018-01-01 -87900.007712 -34728.754839
1  2018-01-02 -87212.648507 -10339.342122
2  2018-01-03 -51144.178016  61185.986604
3  2018-01-04 -84516.287214  20688.783182
4  2018-01-05 -88308.948468  40942.712629
5  2018-01-06 -85455.199578  -3011.495120
6  2018-01-07 -66762.330804   2067.465028
7  2018-01-08 -74048.616190  25232.777676
8  2018-01-09 -80062.733008  27798.387898
9  2018-01-10 -83136.743628  64486.478809
10 2018-01-11 -85227.600627  60019.902539
11 2018-01-12 -67852.333937  43084.654721
13 2018-01-14 -74212.359224  -7600.605278
14 2018-01-15 -85403.059783   4023.884165
15 2018-01-16 -65398.266985   2194.156430
16 2018-01-17 -69946.798184  42354.054804
17 2018-01-18 -68599.372462   9044.134706
18 2018-01-19 -69548.149648  32818.608616
19 2018-01-20 -65355.729661  15388.204609
20 2018-01-21 -54724.708923   6002.500694

H1.Z15 P 저장 완료


In [9]:
# ============================================================
# 7. H1.Z15 I3 전 기간 음수 고착 확인
# ============================================================
df = fetch_daily('H1.Z15', 'I3')
print(f'H1.Z15 I3 전체 기간 max 최대값: {df["max_val"].max():.4f}')
print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
print(f'전체 음수 날짜 수: {(df["max_val"] < 0).sum()}')

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z15 I3 전체 기간 max 최대값: -11.5730
양수 구간 존재 여부: False
전체 음수 날짜 수: 2191


In [10]:
# ============================================================
# 8. V.Z82 I1, I2, I3 음수 구간 확인
# ============================================================
for measurement, threshold in [('I1', -50), ('I2', -100), ('I3', -50)]:
    df = fetch_daily('V.Z82', measurement)
    anomaly = df[df['min_val'] < threshold]
    print(f'V.Z82 {measurement} 이상값 ({len(anomaly)}건):')
    print(anomaly[['day', 'min_val', 'max_val']].head(10).to_string())
    print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
    print()

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z82 I1 이상값 (44건):
          day    min_val     max_val
0  2018-01-01 -88.843368    4.860394
1  2018-01-02 -87.753823   45.247775
3  2018-01-04 -87.098807  100.051087
4  2018-01-05 -86.425189  140.486836
5  2018-01-06 -82.290719   59.718844
6  2018-01-07 -52.033948   69.580952
7  2018-01-08 -67.404247  108.679736
8  2018-01-09 -67.866045  105.261720
9  2018-01-10 -83.052626  174.317126
10 2018-01-11 -83.282534  161.882948
양수 구간 존재 여부: True



/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z82 I2 이상값 (73건):
          day     min_val    max_val
1  2018-01-02 -107.078979 -33.441490
2  2018-01-03 -173.913224 -76.206162
3  2018-01-04 -137.613673 -40.918031
4  2018-01-05 -157.677907 -41.892238
5  2018-01-06 -111.901332 -43.798553
6  2018-01-07 -116.046195 -53.538509
7  2018-01-08 -134.920278 -45.984436
8  2018-01-09 -132.272690 -43.328769
9  2018-01-10 -179.236615 -39.514018
10 2018-01-11 -169.289015 -42.238274
양수 구간 존재 여부: True



/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z82 I3 이상값 (1건):
          day   min_val   max_val
72 2018-03-14 -50.11237  5.525827
양수 구간 존재 여부: True



In [11]:
# ============================================================
# 9. V.ZE84 I1, I2, I3 전체 음수 확인
# ============================================================
for measurement in ['I1', 'I2', 'I3']:
    df = fetch_daily('V.ZE84', measurement)
    print(f'V.ZE84 {measurement} max 최대값: {df["max_val"].max():.4f}')
    print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
    print(f'데이터 시작일: {df["day"].min().date()}, 종료일: {df["day"].max().date()}')
    print()

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.ZE84 I1 max 최대값: -0.4559
양수 구간 존재 여부: False
데이터 시작일: 2022-11-23, 종료일: 2023-12-31

V.ZE84 I2 max 최대값: -0.4591
양수 구간 존재 여부: False
데이터 시작일: 2022-11-23, 종료일: 2023-12-31

V.ZE84 I3 max 최대값: -0.4651
양수 구간 존재 여부: False
데이터 시작일: 2022-11-23, 종료일: 2023-12-21



/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


In [12]:
# ============================================================
# 10. V.Z84 I1, I2, I3 전체 음수 확인
# ============================================================
for measurement in ['I1', 'I2', 'I3']:
    df = fetch_daily('V.Z84', measurement)
    print(f'V.Z84 {measurement} max 최대값: {df["max_val"].max():.4f}')
    print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
    print(f'데이터 시작일: {df["day"].min().date()}, 종료일: {df["day"].max().date()}')
    print()

/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z84 I1 max 최대값: 0.0000
양수 구간 존재 여부: False
데이터 시작일: 2019-06-29, 종료일: 2023-12-31



/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z84 I2 max 최대값: 0.0000
양수 구간 존재 여부: False
데이터 시작일: 2019-06-29, 종료일: 2023-12-31



/tmp/ipykernel_90421/3673009491.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z84 I3 max 최대값: 0.0000
양수 구간 존재 여부: False
데이터 시작일: 2019-06-29, 종료일: 2023-12-31

